[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NASA-Gradient-Descent-Farmers/NASA-warm-up/blob/sant-calibration/Notebooks/rotation_tool.ipynb)

# `comparar_rotaciones`: comparing crop rotations anywhere in Mexico

* **Author:** Santiago (Gradient Descent Farmers) · **Branch:** `sant-calibration` · **Code:** `src/rotations/` (see its README)
* **What it does:** takes a farmer's place, current rotation, water regime and priorities. It returns the current rotation plus three alternatives, each with nine indicators (income, bad-year income, risk, water, nitrogen, carbon, erosion, pest break, forage) and a Spanish card: why, evidence, benefits, risks, gaps, and what to check with a technician.
* **Data:** SIAP district history 2003-2025 (19 crops, both cycles, rainfed and irrigated); NASA POWER climate 2001-2025 for all 2,463 municipalities, rain corrected with CHIRPS; FAO-56 and IPCC crop coefficients.
* **Rule:** alternatives only use crops the district already grows. Every number comes from code, so an LLM only phrases it.

## 0. Setup

In [1]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/NASA-Gradient-Descent-Farmers/NASA-warm-up.git"
BRANCH = "sant-calibration"
if "google.colab" in sys.modules:
    if not Path("/content/NASA-warm-up").exists():
        !git clone -q --branch {BRANCH} {REPO_URL} /content/NASA-warm-up
    !pip install -q pyarrow
    ROOT = Path("/content/NASA-warm-up")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "rotations").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

The tool needs six small tables (~10 MB). Either unzip the shared data package into `data/shared/`, or build them: `python -m src.rotations.build` downloads SIAP (23 files) and NASA POWER (~450 cells) the first time, which takes a while, and is ~3 min after that.

In [2]:
from src.rotations import siap_stats
try:
    siap_stats.load_districts()
    print("tables found")
except FileNotFoundError:
    !python -m src.rotations.build

tables found


/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd
from src.rotations.compare import comparar_rotaciones
pd.set_option("display.width", 200, "display.max_colwidth", 60)

def table(result):
    rows = {}
    for r in result["rotaciones"]:
        rows[r["nombre"]] = {k: f"{v['texto']}  [{v['puntaje']} · {v['evidencia']}]" for k, v in r["indicadores"].items()}
        rows[r["nombre"]]["puntaje con prioridades"] = r["puntaje_prioridades"]
    return pd.DataFrame(rows)

def show_card(r):
    t = r["tarjeta"]
    print(f"### {r['nombre']}  —  {r['secuencia']}")
    print("Por qué:", t["por_que"])
    for title, key in [("Evidencia", "evidencia"), ("Beneficios", "beneficios"), ("Riesgos", "riesgos"),
                       ("Vacíos de información", "vacios"), ("Revisar con un técnico", "revisar_con_tecnico")]:
        items = [x["texto"] + f" ({x['fuente']}, evidencia {x['evidencia']})" if isinstance(x, dict) else x for x in t[key]]
        if items:
            print(f"{title}:"); [print("  •", x) for x in items]
    print("Meses con suelo descubierto:", t["meses_descubiertos"], "\n")

## 1. Valles Centrales, Oaxaca: rainfed maize, priorities risk and soil

A smallholder in Villa de Zaachila sows maize every year, grazes the stover and keeps some animals.

In [4]:
oax = comparar_rotaciones("Villa de Zaachila", "maíz", estado="Oaxaca", agua="temporal",
                          prioridades=["riesgo", "suelo"], residuo="pastoreo", tiene_animales=True)
print(oax["lugar"]); print("\n".join(oax["tu_lugar"]))
table(oax)

{'municipio': 'Villa de Zaachila', 'distrito': 'Valles Centrales', 'estado': 'Oaxaca', 'agua': 'temporal', 'notas': []}
Temporada de lluvias (meses en que la lluvia alcanza al menos la mitad de la evaporación): jun a oct.
Lluvia anual típica ~813 mm; en un año seco (1 de cada 5) ~706 mm.
Meses con riesgo de helada (en 2 de cada 10 años o más hay una noche ≤2 °C): ene, nov, dic.
Hay canícula: julio-agosto llueve menos que junio y septiembre.
Supuse que siembras en jun (inicio de las lluvias). Si siembras en otro mes, dímelo para ajustar heladas, agua y calendario.
Con tu rotación actual la parcela queda descubierta en: ene, feb, mar, abr, may, jun.
Ojo: en may, jun llueve fuerte y el suelo puede estar descubierto.
Maíz: en 24% de los años hubo al menos una noche ≤2 °C mientras el cultivo estaría en campo (sembrando en junio)


,Tu rotación actual: maíz,Con cultivo de invierno: maíz – garbanzo,Con leguminosa: maíz – frijol,Milpa: milpa (maíz-frijol-calabaza)
ingreso,"$4,019 [10 · alta]","$14,643 [37 · alta]","$6,824 [17 · alta]","$8,563 [21 · baja]"
ingreso_p10,"$2,306 [8 · alta]","$10,555 [35 · alta]","$5,042 [17 · alta]","$5,868 [20 · baja]"
riesgo,1.3 de cada 10 [67 · alta],0.7 de cada 10 [84 · alta],1.1 de cada 10 [73 · alta],0.0 de cada 10 [100 · baja]
agua,65% [50 · media],61% [44 · media],71% [58 · media],66% [51 · media]
nitrogeno,-20 kg N/ha [63 · baja],-22 kg N/ha [61 · baja],-15 kg N/ha [66 · baja],-25 kg N/ha [60 · baja]
carbono,0.15 t C/ha [15 · baja],0.32 t C/ha [32 · baja],0.14 t C/ha [14 · baja],0.19 t C/ha [19 · baja]
erosion,36% [40 · media],36% [40 · media],35% [41 · media],36% [40 · media]
plagas,0% [0 · regla],100% [100 · regla],100% [100 · regla],50% [50 · regla]
forraje,0.4 t MS/ha [5 · media],1.1 t MS/ha [14 · media],0.5 t MS/ha [6 · media],0.5 t MS/ha [7 · media]
puntaje con prioridades,30,50,43,42


In [5]:
for r in oax["rotaciones"]:
    show_card(r)
print("Cultivos de nicho:", oax["cultivos_de_nicho"]["cultivos"])

### Tu rotación actual: maíz  —  Año 1: maíz (cada año)
Por qué: Es lo que siembras hoy. Sirve de punto de comparación.
Evidencia:
  • Maíz (primavera-verano, temporal): en el distrito Valles Centrales hay 23 años con registro y ~149,683 ha sembradas por año. Produce 1.07 t por ha sembrada (últimos 10 años, contando lo que se pierde) a $4,410/t (precio medio rural, últimos 5 años). (SIAP, evidencia alta)
Riesgos:
  • Maíz: en 24% de los años hubo al menos una noche ≤2 °C mientras el cultivo estaría en campo (sembrando en junio)
Meses con suelo descubierto: ene, feb, mar, abr, may, jun 

### Con cultivo de invierno: maíz – garbanzo  —  Año 1: maíz y después garbanzo (otoño-invierno) (cada año)
Por qué: Agrega garbanzo en otoño-invierno, después de la cosecha, aprovechando la humedad que queda en el suelo. El terreno deja de quedar descubierto en invierno.
Evidencia:
  • Maíz (primavera-verano, temporal): en el distrito Valles Centrales hay 23 años con registro y ~149,683 ha sembradas po

## 2. Celaya, Guanajuato: the same question in the Bajío

Rainfed maize with stover left in the field. Here the district also grows chickpea in autumn-winter on residual moisture, so the tool can propose a second crop.

In [6]:
cel = comparar_rotaciones("Celaya", "maíz", estado="Guanajuato", prioridades=["agua", "ingreso"])
print("\n".join(cel["tu_lugar"]))
table(cel)

Temporada de lluvias (meses en que la lluvia alcanza al menos la mitad de la evaporación): jun a sep.
Lluvia anual típica ~663 mm; en un año seco (1 de cada 5) ~512 mm.
Meses con riesgo de helada (en 2 de cada 10 años o más hay una noche ≤2 °C): ene, feb, nov, dic.
Supuse que siembras en jun (inicio de las lluvias). Si siembras en otro mes, dímelo para ajustar heladas, agua y calendario.
Con tu rotación actual la parcela queda descubierta en: ene, feb, mar, abr, may, jun, dic.
Ojo: en jun llueve fuerte y el suelo puede estar descubierto.


,Tu rotación actual: maíz,Con cultivo de invierno: maíz – garbanzo,Con leguminosa: maíz – frijol,Milpa: milpa (maíz-frijol-calabaza)
ingreso,"$9,122 [23 · alta]","$16,505 [41 · alta]","$8,978 [22 · alta]","$12,710 [32 · baja]"
ingreso_p10,"$2,979 [10 · alta]","$13,583 [45 · alta]","$2,987 [10 · alta]","$5,201 [17 · baja]"
riesgo,1.7 de cada 10 [57 · alta],1.2 de cada 10 [69 · alta],2.0 de cada 10 [51 · alta],1.9 de cada 10 [52 · baja]
agua,56% [37 · media],56% [37 · media],60% [42 · media],57% [39 · media]
nitrogeno,-25 kg N/ha [60 · baja],-20 kg N/ha [62 · baja],-14 kg N/ha [66 · baja],-26 kg N/ha [59 · baja]
carbono,0.30 t C/ha [30 · baja],0.50 t C/ha [50 · baja],0.25 t C/ha [25 · baja],0.34 t C/ha [34 · baja]
erosion,27% [55 · media],27% [55 · media],26% [57 · media],27% [55 · media]
plagas,0% [0 · regla],100% [100 · regla],100% [100 · regla],50% [50 · regla]
forraje,0.0 t MS/ha [0 · media],0.0 t MS/ha [0 · media],0.0 t MS/ha [0 · media],0.0 t MS/ha [0 · media]
puntaje con prioridades,28,46,39,36


In [7]:
show_card(cel["rotaciones"][1])

### Con cultivo de invierno: maíz – garbanzo  —  Año 1: maíz y después garbanzo (otoño-invierno) (cada año)
Por qué: Agrega garbanzo en otoño-invierno, después de la cosecha, aprovechando la humedad que queda en el suelo. El terreno deja de quedar descubierto en invierno.
Evidencia:
  • Maíz (primavera-verano, temporal): en el distrito Celaya hay 23 años con registro y ~53,874 ha sembradas por año. Produce 1.77 t por ha sembrada (últimos 10 años, contando lo que se pierde) a $5,989/t (precio medio rural, últimos 5 años). (SIAP, evidencia alta)
  • Garbanzo (otoño-invierno, temporal): en el distrito Celaya hay 14 años con registro y ~480 ha sembradas por año. Produce 0.87 t por ha sembrada (últimos 10 años, contando lo que se pierde) a $8,231/t (precio medio rural, últimos 5 años). (SIAP, evidencia alta)
Beneficios:
  • Más ingreso: +$6,347 por ha al año en promedio frente a tu rotación actual (rango 90%: +$5,399 a +$7,183; 14 años de SIAP)
  • Más ingreso en un año malo (+$10,604)
  • 

## 3. Irrigated maize in Culiacán, Sinaloa

In Sinaloa, irrigated maize is an autumn-winter crop, and the tool places it there. The water indicator becomes net irrigation need in mm (m³/ha).

In [8]:
sin = comparar_rotaciones("Culiacán", "maíz", estado="Sinaloa", agua="riego", prioridades=["agua"])
print("\n".join(sin["tu_lugar"]))
table(sin)

Temporada de lluvias (meses en que la lluvia alcanza al menos la mitad de la evaporación): jul a sep.
Lluvia anual típica ~607 mm; en un año seco (1 de cada 5) ~527 mm.
Meses con calor fuerte (3 días o más ≥35 °C en promedio): mar, abr, may, jun, jul, ago, oct.
Con tu rotación actual la parcela queda descubierta en: abr, may, jun, jul, ago, sep, oct, nov.
Ojo: en jul, ago, sep llueve fuerte y el suelo puede estar descubierto.


,Tu rotación actual: maíz,Con leguminosa: maíz – frijol,Menos agua de riego: cártamo – maíz,Con cultivo de verano: sorgo forrajero – maíz
ingreso,"$64,015 [100 · alta]","$48,189 [100 · alta]","$30,944 [77 · alta]","$72,034 [100 · alta]"
ingreso_p10,"$57,705 [100 · alta]","$42,640 [100 · alta]","$23,699 [79 · alta]","$65,830 [100 · alta]"
riesgo,0.4 de cada 10 [89 · alta],0.4 de cada 10 [89 · alta],1.3 de cada 10 [67 · alta],0.6 de cada 10 [85 · alta]
agua,"404 mm (≈4,041 m³/ha) [49 · media]","376 mm (≈3,761 m³/ha) [53 · media]","343 mm (≈3,430 m³/ha) [57 · media]","680 mm (≈6,797 m³/ha) [15 · media]"
nitrogeno,-162 kg N/ha [0 · baja],-85 kg N/ha [22 · baja],-93 kg N/ha [17 · baja],-209 kg N/ha [0 · baja]
carbono,1.60 t C/ha [100 · baja],0.95 t C/ha [95 · baja],0.93 t C/ha [93 · baja],1.80 t C/ha [100 · baja]
erosion,50% [17 · media],50% [17 · media],50% [17 · media],41% [32 · media]
plagas,0% [0 · regla],100% [100 · regla],100% [100 · regla],0% [0 · regla]
forraje,0.0 t MS/ha [0 · media],0.0 t MS/ha [0 · media],0.0 t MS/ha [0 · media],3.3 t MS/ha [42 · media]
puntaje con prioridades,46,59,55,40


## 4. Asking about a specific rotation

The farmer can add rotations to compare (`propuestas`). The sowing month changes the frost and water results. Texcoco farmers often sow in April with residual moisture, rather than at the start of the rains.

In [9]:
tex = comparar_rotaciones("Texcoco", "maíz", estado="México", prioridades=["suelo", "forraje"], residuo="sacar",
                          tiene_animales=True, mes_siembra=4, propuestas=["maíz - haba", "maíz - avena forrajera - frijol"])
table(tex)

,Tu rotación actual: maíz,Tu propuesta: maíz – haba,Tu propuesta: maíz – avena forrajera – frijol,Con leguminosa: maíz – frijol,Milpa: milpa (maíz-frijol-calabaza),Con forraje: maíz – avena forrajera
ingreso,"$26,668 [67 · alta]","$31,551 [79 · alta]","$17,515 [44 · alta]","$21,643 [54 · alta]","$29,306 [73 · baja]","$17,964 [45 · alta]"
ingreso_p10,"$20,034 [67 · alta]","$29,280 [98 · alta]","$14,907 [50 · alta]","$17,460 [58 · alta]","$22,609 [75 · baja]","$14,821 [49 · alta]"
riesgo,0.0 de cada 10 [100 · alta],0.0 de cada 10 [100 · alta],0.0 de cada 10 [100 · alta],0.0 de cada 10 [100 · alta],0.0 de cada 10 [100 · baja],0.0 de cada 10 [100 · alta]
agua,64% [49 · media],58% [40 · media],56% [38 · media],62% [45 · media],56% [38 · media],55% [35 · media]
nitrogeno,-70 kg N/ha [31 · baja],-32 kg N/ha [55 · baja],-62 kg N/ha [36 · baja],-44 kg N/ha [47 · baja],-70 kg N/ha [31 · baja],-84 kg N/ha [23 · baja]
carbono,0.34 t C/ha [34 · baja],0.28 t C/ha [28 · baja],0.33 t C/ha [33 · baja],0.23 t C/ha [23 · baja],0.34 t C/ha [34 · baja],0.43 t C/ha [43 · baja]
erosion,13% [78 · media],30% [50 · media],37% [38 · media],16% [74 · media],13% [78 · media],46% [23 · media]
plagas,0% [0 · regla],100% [100 · regla],67% [67 · regla],100% [100 · regla],50% [50 · regla],0% [0 · regla]
forraje,1.6 t MS/ha [20 · media],1.7 t MS/ha [21 · media],2.4 t MS/ha [30 · media],1.3 t MS/ha [17 · media],1.7 t MS/ha [21 · media],3.1 t MS/ha [38 · media]
puntaje con prioridades,39,50,41,46,44,36


## 5. Limits

The tool shows these to every farmer (`vacios_generales`). See `src/rotations/README.md` for the full list and the work that could fix each one.

In [10]:
print("\n".join("• " + g for g in tex["vacios_generales"]))
print("\n" + tex["aviso"])

• El ingreso es el valor de la cosecha: no incluye costos de semilla, fertilizante, maquinaria ni mano de obra.
• SIAP reporta por distrito, no por parcela: tu parcela puede rendir más o menos que el promedio.
• Precios: mediana de los últimos 5 años, sin ajustar por inflación ni proyectar a futuro.
• Todavía no se incluye el efecto de un cultivo sobre el rendimiento del siguiente (p. ej., maíz después de frijol).
• Nitrógeno: balance sin fertilizante ni estiércol, con valores de fijación de estudios internacionales.
• Carbono: sirve para comparar opciones, no para medir el cambio real de materia orgánica (eso requiere análisis de suelo).
• Clima de NASA POWER en celdas de ~50 km: en hondonadas puede helar más y la lluvia de tu parcela puede variar.

Esta comparación usa datos públicos (SIAP, NASA) y supuestos de literatura. Sirve para explorar opciones, no reemplaza tu experiencia ni la de un técnico o extensionista de tu zona.
